# HC3 Unified AI-Only T5 Perturbation

This Colab notebook reads `hc3_unified_10000_seed42_clean.csv` in the HC3 sample format, creates one final full-text perturbed AI answer for each original `chatgpt_answers` entry, and applies up to 20 word-level perturbations inside that answer using batched T5 generation. `human_answers` is left unchanged. It does not mount Google Drive; upload the CSV into the Colab runtime or place it in the notebook's current working directory.


In [ ]:
!pip -q install transformers sentencepiece accelerate pandas tqdm

In [ ]:
import ast
import json
import random
import re
import time
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Upload hc3_unified_10000_seed42_clean.csv to Colab, or change this path.
INPUT_CSV = "hc3_unified_10000_seed42_clean.csv"
OUTPUT_CSV = "hc3_unified_t5_perturbed_ai_clean.csv"
REPORT_JSON = "hc3_unified_t5_perturbed_ai_clean_report.json"

# t5-small is much faster. Change to "google-t5/t5-base" if you want higher-quality fills and can spend more time.
MODEL_NAME = "google-t5/t5-small"
RANDOM_SEED = 42
MASK_LEN = 1
PERTURBATIONS_PER_ANSWER = 20
GENERATION_BATCH_SIZE = 32
GENERATION_NUM_BEAMS = 1
GENERATION_MAX_NEW_TOKENS = 8
MAX_INPUT_WORDS_FOR_MASK = 256
MAX_ROWS = None  # Set to a small integer for a smoke test.
CHECKPOINT_EVERY = 100

random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)


In [ ]:
input_path = Path(INPUT_CSV)
if not input_path.exists():
    try:
        from google.colab import files
    except ModuleNotFoundError as exc:
        raise FileNotFoundError(
            f"Input CSV was not found at {INPUT_CSV}. Set INPUT_CSV to an existing local path."
        ) from exc

    print(f"Input CSV was not found at {INPUT_CSV}. Upload hc3_unified_10000_seed42_clean.csv now.")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No CSV was uploaded.")
    uploaded_name = next(iter(uploaded.keys()))
    INPUT_CSV = uploaded_name

print("Using input CSV:", INPUT_CSV)


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Loading model:", MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_kwargs = {"torch_dtype": torch.float16} if device == "cuda" else {}
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, **model_kwargs)
model = model.to(device)
model.eval()
print("Loaded on:", device)


In [ ]:
EXPECTED_COLUMNS = ["hc3_row_id", "source", "question", "human_answers", "chatgpt_answers"]


def parse_answer_list(value):
    if isinstance(value, list):
        return [str(item) for item in value if item is not None]
    if value is None or pd.isna(value):
        return []

    text = str(value).strip()
    if not text:
        return []

    for parser in (json.loads, ast.literal_eval):
        try:
            parsed = parser(text)
            if isinstance(parsed, list):
                return [str(item) for item in parsed if item is not None]
            if parsed is None:
                return []
            return [str(parsed)]
        except Exception:
            pass

    return [text]


def dump_answer_list(values):
    return json.dumps([str(item) for item in values if item is not None], ensure_ascii=False)


def extract_t5_fill(decoded_text):
    match = re.search(r"<extra_id_0>\s*(.*?)\s*(<extra_id_\d+>|</s>|$)", decoded_text)
    if not match:
        return ""
    fill = match.group(1).strip()
    fill = re.sub(r"\s+", " ", fill)
    return fill


def generate_fills(masked_texts):
    fills = []
    decoded_outputs = []
    for start in range(0, len(masked_texts), int(GENERATION_BATCH_SIZE)):
        chunk = masked_texts[start : start + int(GENERATION_BATCH_SIZE)]
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                do_sample=False,
                num_beams=int(GENERATION_NUM_BEAMS),
                max_new_tokens=int(GENERATION_MAX_NEW_TOKENS),
            )

        decoded_chunk = tokenizer.batch_decode(outputs, skip_special_tokens=False)
        decoded_outputs.extend(decoded_chunk)
        fills.extend(extract_t5_fill(decoded) for decoded in decoded_chunk)
    return fills, decoded_outputs


def select_mask_starts(words, mask_len=MASK_LEN, max_perturbations=PERTURBATIONS_PER_ANSWER):
    if len(words) <= mask_len + 1:
        return []
    max_start = len(words) - mask_len
    if MAX_INPUT_WORDS_FOR_MASK is not None:
        max_start = min(max_start, max(0, int(MAX_INPUT_WORDS_FOR_MASK) - mask_len))
    candidate_starts = list(range(max_start + 1))
    random.shuffle(candidate_starts)
    return candidate_starts[: min(int(max_perturbations), len(candidate_starts))]


def perturb_answer(text, max_perturbations=PERTURBATIONS_PER_ANSWER):
    original = str(text or "")
    words = original.split()
    starts = select_mask_starts(words, max_perturbations=max_perturbations)
    if not starts:
        return original, {"changed": False, "changes_applied": 0, "attempts": 0, "reason": "too_short", "spans": []}

    masked_texts = []
    masked_metadata = []
    for start in starts:
        end = start + MASK_LEN
        masked_words = words[:start] + ["<extra_id_0>"] + words[end:]
        masked_text = " ".join(masked_words)
        masked_texts.append(masked_text)
        masked_metadata.append({"mask_start": int(start), "mask_len": int(MASK_LEN), "masked_text": masked_text})

    fills, decoded_outputs = generate_fills(masked_texts)
    replacements = []
    span_logs = []
    for metadata, fill, decoded in zip(masked_metadata, fills, decoded_outputs):
        start = int(metadata["mask_start"])
        end = start + int(metadata["mask_len"])
        original_span = " ".join(words[start:end])
        changed = bool(fill) and fill != original_span
        if changed:
            replacements.append((start, end, fill.split()))
        span_logs.append({
            **metadata,
            "fill": fill,
            "decoded": decoded,
            "changed": changed,
        })

    perturbed_words = list(words)
    for start, end, fill_words in sorted(replacements, key=lambda item: item[0], reverse=True):
        perturbed_words[start:end] = fill_words
    perturbed = " ".join(perturbed_words)

    changed_count = len(replacements)
    return perturbed, {
        "changed": changed_count > 0,
        "changes_applied": changed_count,
        "attempts": len(starts),
        "spans": span_logs,
    }


In [ ]:
df = pd.read_csv(INPUT_CSV)
missing = [column for column in EXPECTED_COLUMNS if column not in df.columns]
if missing:
    raise ValueError(f"Input CSV is missing HC3 sample columns: {missing}")

if MAX_ROWS is not None:
    df = df.head(int(MAX_ROWS)).copy()

# Keep only the HC3 sample columns so the output is compatible with the project pipeline.
out_df = df[EXPECTED_COLUMNS].copy()

stats = {
    "input_csv": INPUT_CSV,
    "output_csv": OUTPUT_CSV,
    "model_name": MODEL_NAME,
    "rows_total": int(len(out_df)),
    "ai_rows_with_answers": 0,
    "ai_answers_total": 0,
    "ai_answers_changed": 0,
    "perturbation_steps_total": 0,
    "perturbation_steps_changed": 0,
    "human_answers_changed": 0,
    "started_at_unix": time.time(),
    "config": {
        "random_seed": RANDOM_SEED,
        "mask_len": MASK_LEN,
        "perturbations_per_answer": PERTURBATIONS_PER_ANSWER,
        "generation_batch_size": GENERATION_BATCH_SIZE,
        "generation_num_beams": GENERATION_NUM_BEAMS,
        "generation_max_new_tokens": GENERATION_MAX_NEW_TOKENS,
        "max_input_words_for_mask": MAX_INPUT_WORDS_FOR_MASK,
        "max_rows": MAX_ROWS,
    },
}

def format_seconds(seconds):
    seconds = max(int(seconds), 0)
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


progress = tqdm(out_df.iterrows(), total=len(out_df), desc="Perturbing AI rows", unit="row", dynamic_ncols=True)
for row_number, (row_index, row) in enumerate(progress, start=1):
    human_answers = parse_answer_list(row["human_answers"])
    ai_answers = parse_answer_list(row["chatgpt_answers"])

    perturbed_ai_answers = []
    if ai_answers:
        stats["ai_rows_with_answers"] += 1
        for base_answer in ai_answers:
            perturbed, info = perturb_answer(base_answer)
            perturbed_ai_answers.append(perturbed)
            stats["ai_answers_total"] += 1
            if info.get("changed"):
                stats["ai_answers_changed"] += 1
            stats["perturbation_steps_total"] += int(info.get("attempts", 0))
            stats["perturbation_steps_changed"] += int(info.get("changes_applied", 0))

    # Human answers are intentionally not perturbed; they are only normalized back to JSON-list strings.
    out_df.at[row_index, "human_answers"] = dump_answer_list(human_answers)
    out_df.at[row_index, "chatgpt_answers"] = dump_answer_list(perturbed_ai_answers)

    elapsed = time.time() - stats["started_at_unix"]
    seconds_per_row = elapsed / max(row_number, 1)
    eta_seconds = seconds_per_row * max(len(out_df) - row_number, 0)
    progress.set_postfix(
        eta=format_seconds(eta_seconds),
        steps=stats["perturbation_steps_total"],
        changed=stats["perturbation_steps_changed"],
        refresh=False,
    )

    if CHECKPOINT_EVERY and row_number % int(CHECKPOINT_EVERY) == 0:
        out_df.to_csv(OUTPUT_CSV, index=False)

stats["finished_at_unix"] = time.time()
stats["elapsed_seconds"] = stats["finished_at_unix"] - stats["started_at_unix"]

out_df.to_csv(OUTPUT_CSV, index=False)
Path(REPORT_JSON).write_text(json.dumps(stats, indent=2), encoding="utf-8")

print("Wrote:", OUTPUT_CSV)
print("Wrote:", REPORT_JSON)
print(json.dumps(stats, indent=2))


In [ ]:
preview = pd.read_csv(OUTPUT_CSV)
print(preview.shape)
preview.head(3)


In [ ]:
# Optional Colab download. This does not use Google Drive.
try:
    from google.colab import files
    files.download(OUTPUT_CSV)
    files.download(REPORT_JSON)
except Exception as exc:
    print("Skipping automatic download:", exc)
